# 04.10 - Supervised Learning: Klasifikasi Akurasi Tinggi dengan SVM (K-Means DE)

**Tujuan Analisis:**
Berdasarkan paper referensi (Wang, 2025), *Support Vector Machine* (SVM) dengan Kernel RBF direkomendasikan untuk sistem prediksi klasifikasi pelanggan baru karena menawarkan keseimbangan terbaik antara akurasi yang tinggi dan kecepatan komputasi.

Berbeda dengan *Decision Tree*, model ini bersifat *Black-Box*. Pada tahap ini, SVM digunakan untuk mengklasifikasikan cluster pelanggan yang dihasilkan oleh **K-Means DE** (Differential Evolution).

**Batasan Komputasi:**
Algoritma SVM memiliki kompleksitas waktu yang kuadratik terhadap jumlah sampel. Untuk menghindari beban komputasi yang terlalu berat, **ukuran sampel data latih dibatasi maksimal 2.000 baris**.

**Alur Kerja (Input & Output):**
* **Input:** Dataset `hasildata_kmeans-de.csv`.
* **Proses Wajib:** *Scaling* menggunakan `StandardScaler` agar algoritma tidak bias terhadap fitur dengan angka besar.
* **Output:** Model klasifikasi siap *deploy* dengan akurasi maksimal.

In [1]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. BACA DATA (Gunakan Pemenang Clustering: Standard K-Means)
df = pd.read_csv('../data/Labeled/hasildata_kmeans-de.csv')

fitur = [f'Var{i}' for i in range(1, 12)]
X = df[fitur]
y = df['Cluster']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Membatasi sampel data latih maksimal 2000 baris untuk mencegah processing delays
X_train_svm = X_train[:2000]
y_train_svm = y_train[:2000]

# 2. SCALING DATA (Langkah wajib untuk algoritma SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_svm)
X_test_scaled = scaler.transform(X_test)

# 3. TRAINING MODEL SVM
model_svm = SVC(kernel='rbf', random_state=42)
model_svm.fit(X_train_scaled, y_train_svm)

# Evaluasi
prediksi_svm = model_svm.predict(X_test_scaled)
print("=== CLASSIFICATION REPORT: SVM (DE K-MEANS) ===\n")
print(classification_report(y_test, prediksi_svm))

# 4. EXPORT SCALER & MODEL KE FOLDER 'models'
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler_svm_de.pkl')
joblib.dump(model_svm, '../models/model_svm_classification_de.pkl')
print("\n[SUCCESS] Scaler & Model SVM diekspor ke folder '../models/'!")

=== CLASSIFICATION REPORT: SVM (DE K-MEANS) ===

              precision    recall  f1-score   support

           0       0.94      0.93      0.94       138
           1       0.89      0.86      0.87       165
           2       0.97      0.92      0.94       165
           3       0.92      0.99      0.95        84
           4       0.95      0.80      0.87       111
           5       0.80      0.90      0.85       204

    accuracy                           0.90       867
   macro avg       0.91      0.90      0.90       867
weighted avg       0.90      0.90      0.90       867


[SUCCESS] Scaler & Model SVM diekspor ke folder '../models/'!
